# Play Store Review Sentiment Analysis

This notebook explores the Play Store reviews dataset, trains the required Naive Bayes variants, compares Random Forest and additional studied models, and summarizes the final model selection outcome.

Shared training logic lives in `src/app.py` so the notebook and script stay aligned.

In [ ]:
from pathlib import Path

import json
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import Markdown, display

from app import (
    DATA_PATH,
    METRICS_PATH,
    REPORT_PATH,
    load_and_validate_data,
    preprocess_reviews,
    split_data,
    train_baseline_models,
    run_enhanced_cv,
    select_final_model,
    run_training_pipeline,
    predict_review,
)

PROJECT_ROOT = Path("..").resolve()

## 1. Load and validate the dataset

In [ ]:
raw_df = load_and_validate_data(DATA_PATH)
df = preprocess_reviews(raw_df)

print(f"Rows: {len(df)}")
print(f"Columns: {df.columns.tolist()}")
print("\nClass distribution:")
print(df["polarity"].value_counts().sort_index())

df.head()

## 2. Preprocessing and train/test split

Reviews are cleaned with:

```python
df["review"] = df["review"].str.strip().str.lower()
```

In [ ]:
x_train_text, x_test_text, y_train, y_test = split_data(df)

print(f"Training samples: {len(x_train_text)}")
print(f"Test samples: {len(x_test_text)}")
print(f"Training class distribution:\n{y_train.value_counts().sort_index()}")

## 3. Baseline model comparison (CountVectorizer)

Train `GaussianNB`, `MultinomialNB`, `BernoulliNB`, `RandomForestClassifier`, `LogisticRegression`, and `LinearSVC` on the same 80/20 split.

In [ ]:
baseline_metrics, _, best_nb_name = train_baseline_models(
    x_train_text,
    x_test_text,
    y_train,
    y_test,
)

baseline_df = pd.DataFrame([metric.__dict__ for metric in baseline_metrics])
baseline_display = baseline_df[
    ["model_name", "accuracy", "precision", "recall", "f1"]
].sort_values(["accuracy", "f1"], ascending=False)

print(f"Best Naive Bayes: {best_nb_name}")
baseline_display

## 4. Enhanced cross-validation and holdout diagnostics

Compare Count/TF-IDF vectorizers with unigram and bigram ranges. Model selection uses 5-fold stratified cross-validation on the training split only. The displayed table contains only holdout accuracy, precision, recall, and F1 for every tuned candidate; these holdout values do not determine the winner.

In [ ]:
cv_df, best_cv_candidate = run_enhanced_cv(
    x_train_text,
    y_train,
    x_test_text,
    y_test,
)

candidate_results = (
    cv_df[
        [
            "candidate",
            "holdout_accuracy",
            "holdout_precision",
            "holdout_recall",
            "holdout_f1",
        ]
    ]
    .sort_values(["holdout_accuracy", "holdout_f1"], ascending=False)
    .reset_index(drop=True)
)
candidate_results

## 5. Final model selection and artifacts

Run the full pipeline to persist the best model, JSON metrics, and Markdown report.

In [ ]:
payload = run_training_pipeline(overwrite_artifacts=True)
final_metrics = payload["final_model_metrics"]

print(payload["selection_rationale"])
print(
    f"Final model: {final_metrics['model_name']} | "
    f"F1={final_metrics['f1']:.4f} | accuracy={final_metrics['accuracy']:.4f}"
)

In [ ]:
matrix = final_metrics["confusion_matrix"]
plt.figure(figsize=(6, 5))
sns.heatmap(
    matrix,
    annot=True,
    fmt="d",
    cmap="Blues",
    xticklabels=["Negative (0)", "Positive (1)"],
    yticklabels=["Negative (0)", "Positive (1)"],
)
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.title("Final Model Confusion Matrix")
plt.tight_layout()
plt.show()

## 6. Conclusion

Naive Bayes assumes conditional independence between words, which is a simplifying assumption for text sentiment. `MultinomialNB` is usually the strongest Naive Bayes baseline for count-based text features, and this dataset confirmed that pattern on the required baseline comparison.

Linear models such as Logistic Regression and Linear SVM often perform well on sparse bag-of-words features because they can learn flexible feature weights without the independence assumption. After enhanced tuning with TF-IDF and cross-validation, the final selected model should be treated as the production artifact because it achieved the strongest holdout positive-class F1 on the untouched test set.

In [ ]:
with METRICS_PATH.open("r", encoding="utf-8") as file:
    saved_metrics = json.load(file)

assert saved_metrics["final_model_metrics"]["model_name"] == final_metrics["model_name"]
assert saved_metrics["best_naive_bayes"] == payload["best_naive_bayes"]

positive_review = "this app is amazing and works perfectly every day"
negative_review = "the app keeps crashing and is completely unusable"

print("Sample predictions:")
print("Positive review ->", predict_review(positive_review))
print("Negative review ->", predict_review(negative_review))

display(Markdown(REPORT_PATH.read_text(encoding="utf-8")[:1500] + "\n\n..."))